# P07 SkyOps — Week 10: Streaming Bronze with Auto Loader

**Project:** SkyOps Airline Delay Command Center  
**Project ID:** P07  
**Week:** 10 — Make the Pipeline Move  
**Platform:** Databricks + Delta Lake + Structured Streaming + Auto Loader

## Purpose
This notebook extends the Week 3–7 SkyOps batch pipeline into **streaming ingestion**.

The supplied project pack contains six controlled newline-delimited JSON drops under `streaming/`. They are intentionally staged as separate arrivals so the notebook can demonstrate:

- incremental file discovery with **Auto Loader**
- **Structured Streaming** with a streaming DataFrame
- **checkpointing** and restart-safe progress
- streaming Bronze that preserves what actually arrived
- duplicate event detection
- late / out-of-order event observation
- invalid-reference observation
- schema-drift observation
- reconciliation and audit evidence

> **Important project boundary:** this is a **simulated streaming exercise**, not a real airline feed.

### Week 10 principle
**Do not read all six JSON files together as one batch.** Stage them into the landing folder one at a time and let the same streaming query incrementally discover each new file.

## 1. Week 10 learning objectives
By the end of this notebook you should be able to:

1. Explain batch processing vs streaming.
2. Explain what an event and a stream are.
3. Explain how Auto Loader discovers newly arriving files.
4. Explain why a checkpoint is required.
5. Create a streaming DataFrame from JSON event files.
6. Write incoming events to a Delta Streaming Bronze table.
7. Observe duplicates without silently deleting them.
8. Identify out-of-order / late-arriving events.
9. Identify invalid reference values without corrupting Bronze.
10. Observe schema drift through metadata / rescued data.

## 2. Project contract carried from earlier weeks
The earlier SkyOps notebooks use:

- Catalog: `workspace`
- Schema: `default`
- Project Volume: `/Volumes/p07-skyops/default/skyops`
- Delta as persistent storage
- Bronze as the first persistent ingestion layer
- Silver/Trusted Silver for governed cleansing and DQ
- Gold only from Trusted Silver

**Week 10 does not replace the Week 6 DQ rules or the Week 7 Gold layer.** Streaming Bronze preserves incoming event records so later layers can make governed decisions.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")

volume_path = "/Volumes/p07-skyops/default/skyops"
source_streaming_path = f"{volume_path}/streaming"
landing_path = f"{volume_path}/streaming_landing"
schema_path = f"{volume_path}/_checkpoints/skyops_streaming_schema"
checkpoint_path = f"{volume_path}/_checkpoints/skyops_streaming_bronze"
bronze_table = "bronze_skyops_streaming_events"

print("Source streaming path:", source_streaming_path)
print("Landing path:", landing_path)
print("Schema path:", schema_path)
print("Checkpoint path:", checkpoint_path)
print("Bronze table:", bronze_table)

## 3. Confirm the supplied streaming drops
The project pack contains these six controlled drops:

| Drop | Scenario |
|---|---|
| `drop_01_normal.json` | normal arrival |
| `drop_02_duplicate.json` | duplicate `event_id` |
| `drop_03_late.json` | late-arriving event scenario |
| `drop_04_out_of_order.json` | event sequence arrives out of order |
| `drop_05_invalid_reference.json` | invalid carrier / flight reference |
| `drop_06_schema_drift.json` | new fields + changed schema version |

The exact supplied files are the source of truth for this notebook.

In [ ]:
display(dbutils.fs.ls(source_streaming_path))

### Expected controlled-pack observations
The supplied ZIP currently contains:

- Drop 01: 5 physical rows / 5 distinct `event_id`
- Drop 02: 3 physical rows / 2 distinct `event_id` — one duplicate
- Drop 03: 2 physical rows / 2 distinct `event_id`
- Drop 04: 3 physical rows / 3 distinct `event_id`
- Drop 05: 2 physical rows / 2 distinct `event_id`
- Drop 06: 2 physical rows / 2 distinct `event_id`

So the six supplied files contain **17 physical event rows and 16 distinct event IDs**.

These counts are specific to the supplied project ZIP; do not replace them with larger illustrative counts from the teaching slides.

## 4. Prepare the landing zone
The lesson is to simulate files **arriving over time**.

1. Keep all six source files in `source_streaming_path`.
2. Keep `landing_path` empty initially.
3. Start the streaming query.
4. Copy only Drop 01 into the landing folder.
5. Observe Bronze.
6. Copy Drop 02, then observe again.
7. Continue through Drop 06.

The reset cell later in the notebook is commented out so normal runs do not destroy checkpoint state.

In [ ]:
dbutils.fs.mkdirs(landing_path)
dbutils.fs.mkdirs(f"{volume_path}/_checkpoints")
print("Landing folder is ready:", landing_path)

## 5. Inspect one source event before streaming
This is only a source inspection. It is **not** the streaming read.

The JSON events contain fields such as `event_id`, `event_type`, `event_time`, `event_sequence_no`, `flight_id`, `carrier_code`, airport codes, status, delay, and source metadata.

In [ ]:
display(spark.read.json(f"{source_streaming_path}/drop_01_normal.json"))

## 6. Create the Auto Loader streaming DataFrame
`cloudFiles` is Databricks Auto Loader.

- `spark.read.json(...)` reads files that are present now.
- `spark.readStream.format("cloudFiles")` continuously discovers **new files** added to the landing folder.

`schemaEvolutionMode = rescue` is used so unexpected/new fields or incompatible values can be retained in `_rescued_data` instead of silently disappearing.

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("event_sequence_no", IntegerType(), True),
    StructField("producer_run_id", StringType(), True),
    StructField("schema_version", StringType(), True),
    StructField("flight_id", StringType(), True),
    StructField("flight_date", StringType(), True),
    StructField("carrier_code", StringType(), True),
    StructField("flight_number", IntegerType(), True),
    StructField("origin_airport_code", StringType(), True),
    StructField("destination_airport_code", StringType(), True),
    StructField("current_airport_code", StringType(), True),
    StructField("scheduled_departure_time", StringType(), True),
    StructField("scheduled_arrival_time", StringType(), True),
    StructField("estimated_departure_time", StringType(), True),
    StructField("estimated_arrival_time", StringType(), True),
    StructField("gate", StringType(), True),
    StructField("status_code", StringType(), True),
    StructField("delay_minutes", IntegerType(), True),
    StructField("cancellation_code", StringType(), True),
    StructField("diversion_flag", IntegerType(), True),
    StructField("event_source", StringType(), True)
])

stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .schema(event_schema)
        .load(landing_path)
)

print("isStreaming:", stream_df.isStreaming)
stream_df.printSchema()

## 7. Add streaming Bronze metadata
Bronze should answer:

> **What actually arrived?**

Therefore this step does **not** deduplicate, correct invalid references, or calculate Gold KPIs.

It only adds ingestion/audit metadata such as ingestion timestamp, source file information, and rescued data.

In [ ]:
bronze_stream_df = (
    stream_df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file_path", F.col("_metadata.file_path"))
    .withColumn("_source_file_name", F.col("_metadata.file_name"))
    .withColumn("_source_file_size", F.col("_metadata.file_size"))
    .withColumn("_source_file_modification_time", F.col("_metadata.file_modification_time"))
)

## 8. Start the streaming Bronze query
`availableNow=True` is useful for this controlled lab because it starts a real Structured Streaming query, processes currently available files, updates Delta Bronze, and then stops.

The **checkpoint remains**, so later runs can continue incrementally.

For the intended teaching sequence, place only one new drop in `landing_path` before each run.

In [ ]:
query = (
    bronze_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()
print("Streaming batch completed.")

## 9. Verify the Streaming Bronze table
Run these queries after each newly staged drop.

The first query answers: **How many physical event records arrived?**

The second answers: **How many unique business event IDs arrived?**

In [ ]:
spark.sql(f"""
SELECT
    COUNT(*) AS physical_rows,
    COUNT(DISTINCT event_id) AS distinct_event_ids,
    COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_physical_rows
FROM {bronze_table}
""")

In [ ]:
spark.sql(f"""
SELECT
    event_type,
    COUNT(*) AS physical_rows,
    COUNT(DISTINCT event_id) AS distinct_event_ids
FROM {bronze_table}
GROUP BY event_type
ORDER BY event_type
""")

## 10. Incremental-file demonstration
After the first run, copy the next source file into the landing folder.

Example:

```python
dbutils.fs.cp(
    source_streaming_path + "/drop_01_normal.json",
    landing_path + "/drop_01_normal.json"
)
```

Then run the streaming query again.

For the next arrival, use `drop_02_duplicate.json`, then `drop_03_late.json`, and so on.

**Do not copy all six files before the first run** if you want to demonstrate incremental discovery clearly.

In [ ]:
# Stage Drop 01 only when you are ready to simulate the first arrival.
# dbutils.fs.cp(
#     f"{source_streaming_path}/drop_01_normal.json",
#     f"{landing_path}/drop_01_normal.json"
# )

## 11. Duplicate event investigation
Drop 02 intentionally contains a repeated `event_id`.

We do **not** delete the duplicate from Streaming Bronze. Instead, we expose it for later DQ handling.

In [ ]:
spark.sql(f"""
SELECT
    event_id,
    COUNT(*) AS physical_occurrences,
    MIN(_ingested_at) AS first_ingested_at,
    MAX(_ingested_at) AS last_ingested_at,
    COLLECT_SET(event_type) AS event_types
FROM {bronze_table}
GROUP BY event_id
HAVING COUNT(*) > 1
ORDER BY physical_occurrences DESC, event_id
""")

## 12. Late-arriving event observation
A streaming pipeline cannot assume arrival order and business/event time are always identical.

Compare `event_time` with `_ingested_at` to distinguish when the event says it happened from when the pipeline received it.

In [ ]:
spark.sql(f"""
SELECT
    event_id,
    flight_id,
    event_type,
    event_time,
    _ingested_at,
    event_sequence_no
FROM {bronze_table}
ORDER BY _ingested_at, event_time
""")

## 13. Out-of-order event investigation
Drop 04 intentionally places sequence number 3 before sequence numbers 1 and 2 in the physical file.

We preserve both producer sequence and ingestion context instead of sorting and overwriting the incoming records.

In [ ]:
spark.sql(f"""
WITH ordered AS (
    SELECT
        flight_id,
        event_id,
        event_sequence_no,
        event_time,
        _ingested_at,
        LAG(event_sequence_no) OVER (
            PARTITION BY flight_id
            ORDER BY _ingested_at
        ) AS previous_sequence_no
    FROM {bronze_table}
)
SELECT *
FROM ordered
WHERE previous_sequence_no IS NOT NULL
  AND event_sequence_no < previous_sequence_no
ORDER BY flight_id, _ingested_at
""")

## 14. Invalid-reference investigation
Drop 05 contains a deliberately invalid reference scenario.

Do not mutate Streaming Bronze because a reference is invalid. Compare incoming values with the project's governed reference tables and surface the failure for later DQ/quarantine handling.

In [ ]:
spark.sql(f"""
SELECT
    b.event_id,
    b.flight_id,
    b.carrier_code,
    b.origin_airport_code,
    b.destination_airport_code,
    CASE WHEN c.carrier_code IS NULL THEN 'INVALID_CARRIER' ELSE 'OK' END AS carrier_reference_status,
    CASE WHEN ao.airport_code IS NULL THEN 'INVALID_ORIGIN' ELSE 'OK' END AS origin_reference_status,
    CASE WHEN ad.airport_code IS NULL THEN 'INVALID_DESTINATION' ELSE 'OK' END AS destination_reference_status
FROM {bronze_table} b
LEFT JOIN trusted_silver_carriers c
    ON UPPER(TRIM(b.carrier_code)) = UPPER(TRIM(c.carrier_code))
LEFT JOIN trusted_silver_airports ao
    ON UPPER(TRIM(b.origin_airport_code)) = UPPER(TRIM(ao.airport_code))
LEFT JOIN trusted_silver_airports ad
    ON UPPER(TRIM(b.destination_airport_code)) = UPPER(TRIM(ad.airport_code))
WHERE c.carrier_code IS NULL
   OR ao.airport_code IS NULL
   OR ad.airport_code IS NULL
ORDER BY b._ingested_at
""")

## 15. Schema-drift investigation
Drop 06 introduces `schema_version = 2.0`, new fields such as `terminal_zone` and `operational_note`, and a `delay_minutes` value represented as text (`"fifteen"`).

Because this stream uses Auto Loader **rescue** mode, unexpected data can be retained in `_rescued_data` rather than silently disappearing.

In [ ]:
spark.sql(f"""
SELECT
    event_id,
    schema_version,
    delay_minutes,
    _rescued_data,
    _source_file_name
FROM {bronze_table}
WHERE schema_version <> '1.0'
   OR _rescued_data IS NOT NULL
ORDER BY _ingested_at
""")

## 16. Streaming Bronze audit summary
This summary gives total physical rows, distinct event IDs, duplicates, schema versions, and event types.

In [ ]:
spark.sql(f"""
SELECT
    COUNT(*) AS total_physical_rows,
    COUNT(DISTINCT event_id) AS distinct_event_ids,
    COUNT(*) - COUNT(DISTINCT event_id) AS duplicate_physical_rows,
    COUNT(DISTINCT schema_version) AS schema_versions_seen,
    COUNT(DISTINCT event_type) AS event_types_seen
FROM {bronze_table}
""")

In [ ]:
spark.sql(f"""
SELECT
    schema_version,
    COUNT(*) AS rows
FROM {bronze_table}
GROUP BY schema_version
ORDER BY schema_version
""")

## 17. Check incremental source-file processing
This query shows which files have reached Bronze.

If files are staged one at a time and the same checkpointed stream is rerun, previously processed files should not be appended again simply because the stream was restarted.

In [ ]:
spark.sql(f"""
SELECT
    _source_file_name,
    COUNT(*) AS rows_received,
    MIN(_ingested_at) AS first_ingested_at,
    MAX(_ingested_at) AS last_ingested_at
FROM {bronze_table}
GROUP BY _source_file_name
ORDER BY first_ingested_at, _source_file_name
""")

## 18. Checkpoint explanation
A checkpoint is the stream's persistent memory.

Without a checkpoint, the pipeline cannot reliably know what it has already processed after a restart.

With the checkpoint:

**new file → discovered → processed → checkpoint updated**

On the next run, Auto Loader continues from the recorded progress instead of treating every old file as new work.

In [ ]:
display(dbutils.fs.ls(checkpoint_path))

## 19. Streaming vs batch comparison for SkyOps

| Batch | Week 10 Streaming |
|---|---|
| Reads a bounded set of files | Watches for newly arriving files |
| Processes the current dataset | Processes newly discovered arrivals |
| No streaming checkpoint required | Checkpoint required for restart-safe progress |
| Week 3–7 historical pipeline | Week 10 operational-event ingestion |

Streaming does not replace batch; it adds a way to process data as it arrives.

## 20. Optional clean reset for a fresh classroom demonstration

**Do not run this during a normal streaming run.**

Use it only when intentionally repeating Week 10 from an empty state.

In [ ]:
# DANGER — RESET CELL
# Run only when you intentionally want a fresh Week 10 classroom run.
#
# spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
# dbutils.fs.rm(landing_path, True)
# dbutils.fs.rm(checkpoint_path, True)
# dbutils.fs.rm(schema_path, True)
# dbutils.fs.mkdirs(landing_path)
#
# print("Week 10 streaming state reset.")

## 21. Week 10 acceptance checklist
- [ ] Project is identified as **P07 SkyOps Airline Delay Command Center**.
- [ ] Streaming is described as a **simulated** project exercise.
- [ ] JSON event files are processed incrementally.
- [ ] Auto Loader (`cloudFiles`) is used.
- [ ] Structured Streaming is used.
- [ ] A persistent checkpoint is configured.
- [ ] Streaming Bronze is Delta.
- [ ] Bronze preserves incoming event records and source metadata.
- [ ] Duplicate `event_id` values are detected, not silently deleted.
- [ ] Late/out-of-order events can be observed.
- [ ] Invalid references are surfaced without modifying Bronze.
- [ ] Schema drift is retained through rescue metadata.
- [ ] Counts are reconciled against the actual supplied project drops.
- [ ] Week 6 DQ/quarantine remains downstream of ingestion.
- [ ] Week 7 Gold remains downstream of Trusted Silver.

## 22. Final Week 10 takeaway
### Data in motion

**New file → Auto Loader → Structured Streaming → Streaming Bronze → observe / preserve → later DQ → Trusted Silver → Gold**

The key lesson is:

> **Streaming means continuously processing what is new while keeping enough metadata and checkpoint state to make the pipeline restartable and traceable.**

For this SkyOps project, the six supplied JSON drops are a controlled simulation. They are not a live airline feed.

### AI transparency note
This notebook was generated from the supplied P07 SkyOps project ZIP, the existing Week 3–7 notebooks, and the supplied Week 10 teaching material. Project-specific names, paths, table conventions, streaming-drop scenarios, and the simulated-streaming boundary were kept aligned to those supplied materials.